In [ ]:
import pandas as pd
import numpy as np
import proplot as pplt


In [ ]:
#mean daily
def daily_mean_all_columns(data, start_year, end_year):
    # Create a date range from start to end year
    date_range = pd.date_range(start=f'{start_year}-01-01', end=f'{end_year}-12-31', freq='D')

    # Add the date range as an index to the data
    data = data.copy()
    data.index = date_range

    # Extract day of year to group by (1 to 366 to handle leap years)
    data['DAY_OF_YEAR'] = data.index.dayofyear

    # Calculate the mean for each day of the year for all columns except 'DAY_OF_YEAR'
    daily_mean_df = data.groupby('DAY_OF_YEAR').mean()

    # Remove the 'DAY_OF_YEAR' column from the final output if it was added
    if 'DAY_OF_YEAR' in daily_mean_df.columns:
        daily_mean_df = daily_mean_df.drop(columns=['DAY_OF_YEAR'])

    return daily_mean_df



In [ ]:
# Mean Monthly Function
def monthly_mean_all_columns(data, start_year, end_year):
    # Create a date range from start to end year
    date_range = pd.date_range(start=f'{start_year}-01-01', end=f'{end_year}-12-31', freq='D')

    # Add the date range as an index to the data
    data = data.copy()
    data.index = date_range

    # Extract month to group by (1 to 12)
    data['MONTH'] = data.index.month

    # Calculate the mean for each month for all columns except 'MONTH'
    monthly_mean_df = data.groupby('MONTH').mean()

    # Remove the 'MONTH' column from the final output if it was added
    if 'MONTH' in monthly_mean_df.columns:
        monthly_mean_df = monthly_mean_df.drop(columns=['MONTH'])

    return monthly_mean_df


In [ ]:
def seasonality(data):
    day_of_year = np.array([16, 45.5, 75, 105.5, 136, 166.5, 197, 228, 258, 289, 319, 350])

    # Calculate angle in degrees, then convert to radians for the sine calculation
    angle = day_of_year / 365 * 360  # angle in degrees
    angle_radians = np.radians(angle)  # Convert to radians

    # Element-wise multiplication of data with np.sin(angle_radians)
    S = np.sum(data * np.sin(angle_radians))
    C = np.sum(data * np.cos(angle_radians))
    PR = (S**2 +C**2)**0.5
    SI = PR/np.sum(data)
    if S > 0 and C > 0:
        c_angle = np.degrees(np.arctan(S/C))
    elif C < 0:
        c_angle = np.degrees(np.arctan(S/C)) + 180
    elif S < 0 and C > 0:
         c_angle = np.degrees(np.arctan(S/C)) + 360
    # List of months with 30° intervals
    months = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

    # Find month based on c_angle
    month_index = int(c_angle // 30) % 12  # Ensure it wraps around at 360°
    c_month = months[month_index]

    return SI, c_angle, c_month

In [ ]:
# File paths
dir_topo = 'D:/yuanqi/new urania partition/camels_600.txt'
topo_path = 'D:/yuanqi/new_pdm/camels_attributes_v2.0/camels_attributes_v2.0/camels_topo.txt'
vege_path = 'D:/yuanqi/new_pdm/camels_attributes_v2.0/camels_attributes_v2.0/camels_vege.txt'
# Read and sort camels_info
camels_info = pd.read_csv(dir_topo, delimiter=';')
camels_info = camels_info.sort_values(by='aridity').reset_index(drop=True)
gauge_id = camels_info['gauge_id'].values.astype(np.int64)

# Read topo_info
topo_info = pd.read_csv(topo_path, delimiter=';')
vege_info = pd.read_csv(vege_path, delimiter=';')


# Merge camels_info with topo_info on gauge_id
camels_info = pd.merge(camels_info, topo_info, on='gauge_id', how='left')

# Merge the result with vege_info on gauge_id
camels_info = pd.merge(camels_info, vege_info, on='gauge_id', how='left')
n2 = len(gauge_id)

second method

In [ ]:
results = []
for ii in range(n2):

    #ii = n1 + jj
    stationid = gauge_id[ii]

    aridity = camels_info[camels_info['gauge_id'] == stationid]['aridity'].iloc[0]
    aridity = round(aridity,3)
    lon = camels_info[camels_info['gauge_id'] == stationid]['gauge_lon'].iloc[0]
    lat = camels_info[camels_info['gauge_id'] == stationid]['gauge_lat'].iloc[0]
    if gauge_id[ii]/1e7 < 1:
        stationid = "%08d"%gauge_id[ii]
    else:
        stationid = gauge_id[ii]
    #print(stationid)
    outd_file = pd.read_csv('D:/yuanqi/1106/sim/' + str(stationid) + '_AI_' + str(aridity) + '_simulated26.csv',index_col=0)

    outd = outd_file[730:]


    pmm_d = np.array(outd_file['prcp'])



    mean_monthly = monthly_mean_all_columns(outd,1987,2014)
    simq = mean_monthly['Qsim']
    qb = mean_monthly['qb']
    ri = mean_monthly['ri']
    rs = mean_monthly['rd']-mean_monthly['ri']
    rd = mean_monthly['rd']
    prcp = mean_monthly['prcp']

    q_si, q_angle, q_month = seasonality(simq)
    qb_si, qb_angle, qb_month = seasonality(qb)
    ri_si, ri_angle, ri_month = seasonality(ri)
    rs_si, rs_angle, rs_month = seasonality(rs)
    prcp_si, prcp_angle, prcp_month = seasonality(prcp)

    qb_percent_si, qb_percent_angle, qb_percent_month = seasonality(qb / simq)
    ri_percent_si, ri_percent_angle, ri_percent_month = seasonality(ri / rd)
    rs_percent_si, rs_percent_angle, rs_percent_month = seasonality(rs / rd)

    results.append({
        'stationid': stationid,
        'aridity': aridity,
        'lon': lon,
        'lat': lat,
        'q_si': q_si,
        'q_angle': q_angle,
        'q_angle_revised': q_angle-90,
        'q_month': q_month,
        'qb_si': qb_si,
        'qb_angle': qb_angle,
        'qb_angle_revised': qb_angle-90,
        'qb_month': qb_month,
        'ri_si': ri_si,
        'ri_angle': ri_angle,
        'ri_angle_revised': ri_angle-90,
        'ri_month': ri_month,
        'rs_si': rs_si,
        'rs_angle': rs_angle,
        'rs_angle_revised': rs_angle-90,
        'rs_month': rs_month,
        'prcp_si': prcp_si,
        'prcp_angle': prcp_angle,
        'prcp_angle_revised': prcp_angle-90,
        'prcp_month': prcp_month,

        'qb_percent_si': qb_percent_si,
        'qb_percent_angle': qb_percent_angle,
        'qb_percent_month': qb_percent_month,
        'ri_percent_si': ri_percent_si,
        'ri_percent_angle': ri_percent_angle,
        'ri_percent_month': ri_percent_month,
        'rs_percent_si': rs_percent_si,
        'rs_percent_angle': rs_percent_angle,
        'rs_percent_month': rs_percent_month
    })

# Convert the results list to a DataFrame
results_df2 = pd.DataFrame(results)
results_df2.to_csv('D:/yuanqi/new urania partition/seasonality_index.csv')

In [ ]:
results = []
for ii in range(n2):

    #ii = n1 + jj
    stationid = gauge_id[ii]

    aridity = camels_info[camels_info['gauge_id'] == stationid]['aridity'].iloc[0]
    aridity = round(aridity,3)

    if gauge_id[ii]/1e7 < 1:
        stationid = "%08d"%gauge_id[ii]
    else:
        stationid = gauge_id[ii]
    #print(stationid)

    pet_file = pd.read_csv('D:/yuanqi/new_pdm/seperatebaseflow/streamflow/'  + str(stationid) + '_baseflow_seperated.csv')
    data_TESTC = pd.read_csv('D:/yuanqi/new_pdm/daymet/'  + str(stationid) + '_lump_cida_forcing_leap.txt', delimiter='\s+',skiprows=3)

    x1_start = np.where((pet_file['YR']  ==1987) & (pet_file['MNTH']==1) & (pet_file['DY']==1))[0]
    x1_end = np.where((pet_file['YR']==2014) & (pet_file['MNTH']==12) & (pet_file['DY']==31))[0]
    x2_start = np.where((data_TESTC['Year']==1987) & (data_TESTC['Mnth']==1) & (data_TESTC['Day']==1))[0][0]
    x2_end = np.where((data_TESTC['Year']==2014) & (data_TESTC['Mnth']==12) & (data_TESTC['Day']==31))[0]
    pet_file = pet_file.loc[np.arange(x1_start,x1_end+1)].reset_index(drop=True)
    data_TESTC = data_TESTC.loc[np.arange(x2_start,x2_end+1)].reset_index(drop=True)
    pet_d = np.array(pet_file['PET'])
    pmm_d = np.array(data_TESTC['prcp(mm/day)'])
    outd = pd.DataFrame(columns=['pet','prcp'])
    outd['pet'] = pet_d
    outd['prcp'] = pmm_d


    mean_monthly = monthly_mean_all_columns(outd,1987,2014)

    pet = mean_monthly['pet']
    prcp = mean_monthly['prcp']


    pet_si, pet_angle, pet_month = seasonality(pet)
    prcp_si, prcp_angle, prcp_month = seasonality(prcp)


    results.append({
        'stationid': stationid,
        'aridity': aridity,

        'pet_angle': pet_angle,
        'pet_month': pet_month,
        'pet_si': pet_si,
        'prcp_angle': prcp_angle,
        'prcp_month': prcp_month,
        'prcp_si': prcp_si

    })

# Convert the results list to a DataFrame
results_df2 = pd.DataFrame(results)
results_df2.to_csv('D:/yuanqi/new urania partition/seasonality_index_climate.csv')